# GreenLight: entrenamiento y evaluación de YOLO11 en Google Colab

Este notebook usa las **anotaciones corregidas que exportaste de CVAT**, con una
clase `vehicle`. Entrena YOLO11n/s/m y guarda checkpoints, gráficos y comparaciones
en tu Drive. No necesita CARLA ni ejecutar código en tu GPU local.

**Preparación:** subir el ZIP corregido a `Mi unidad/GreenLight/datasets/`, abrir
este notebook y elegir **Entorno de ejecución → Cambiar tipo de entorno → GPU**.
El tipo de GPU depende de la disponibilidad de Colab.

Si exportaste solo etiquetas, subir también `greenlight_vehicle_cvat.zip`, que
preparamos originalmente. Ese ZIP auxiliar aporta únicamente imágenes: sus
propuestas de etiquetas no se usan.

Los metadatos de tus 500 imágenes están incluidos. Para una selección distinta
(por ejemplo 1.200), subir su `selection.csv` y configurar `METADATA_CSV`.

Ejecutar las celdas en orden. La prueba final está desactivada inicialmente para
reservar `test` mientras elegís configuraciones usando `val`.


## 1. Instalar la versión del proyecto

Se conserva el PyTorch/CUDA de Colab. No se instalan las dependencias de CARLA.


In [ ]:
import subprocess
import sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "ultralytics==8.3.197"])


## 2. Configurar datos y experimento

`CVAT_ZIP` debe ser la exportación **corregida**, formato Ultralytics YOLO Detection.
`IMAGES_ZIP` se necesita solo cuando la exportación no incluye imágenes.

Con tres o más sesiones, `SPLIT_MODE="auto"` reserva sesiones enteras. Con tu
sesión actual, separa tramos temporales, conserva ambas cámaras de cada tramo
juntas y excluye imágenes cercanas a los límites entre conjuntos diferentes.
Esta última modalidad permite un piloto; no equivale a evaluar nuevas sesiones.

Podés asignar manualmente todos los grupos con `GROUP_SPLITS` para distribuir
condiciones conocidas entre conjuntos. El clima no figura en los metadatos y
el muestreo automático no garantiza cuotas por clima. Revisá la tabla del paso 5.

Usar otro `EXPERIMENT_NAME` al cambiar dataset o parámetros. Para continuar tras
una desconexión, mantener el nombre y ejecutar de nuevo: se recupera `last.pt`
cuando el experimento no terminó y la configuración coincide.


In [ ]:
CVAT_ZIP = "/content/drive/MyDrive/GreenLight/datasets/greenlight_cvat_corregido.zip"
IMAGES_ZIP = ""  # Si faltan imágenes: /content/drive/MyDrive/GreenLight/datasets/greenlight_vehicle_cvat.zip
METADATA_CSV = ""  # Vacío: metadatos incluidos de las 500 imágenes originales

EXPERIMENT_NAME = "vehicle_v1"
MODELS = ["yolo11n.pt", "yolo11s.pt", "yolo11m.pt"]
IMGSZ = 640
BATCH = 8  # Reducir a 4 o 2 si la GPU asignada se queda sin memoria
EPOCHS = 100
PATIENCE = 20
SEED = 12345
WORKERS = 2
OPTIMIZER = "AdamW"
LR0 = 0.001

SPLIT_MODE = "auto"  # auto, session o temporal
GAP_SECONDS = 10.0  # Margen a cada lado del límite temporal entre splits
GROUP_SPLITS = {}  # Ejemplo: {"sesion_...:t01": "train", ...}; asignar TODOS los grupos

RUN_PILOT = True  # Cinco épocas con el primer modelo para medir tiempo/memoria
RUN_TRAINING = True
RESUME_INTERRUPTED = True
RUN_FINAL_TEST = False  # Activar cuando hayas fijado modelos y configuración
DETECTION_CONF = 0.20  # Umbral común para conteo/latencia; ajustar usando val


## 3. Conectar Drive y comprobar GPU

El entrenamiento lee los PNG desde el disco temporal `/content`, evitando miles
de lecturas en Drive. Los checkpoints se escriben en Drive al terminar cada época.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import gc
import hashlib
import json
import os
from pathlib import Path
import re
import shutil
import tempfile
import time
from datetime import datetime, timezone

runtime_dir = Path("/content/greenlight_runtime")
runtime_dir.mkdir(exist_ok=True)
os.environ["YOLO_CONFIG_DIR"] = str(runtime_dir)

import numpy as np
import pandas as pd
import torch
import ultralytics
from ultralytics import YOLO, settings

# Solo logs/artefactos locales y en tu Drive; no activar plataformas de tracking.
settings.update({key: False for key in ("wandb", "mlflow", "clearml", "comet", "neptune", "hub", "dvc") if key in settings})
if not torch.cuda.is_available():
    raise RuntimeError("Seleccionar GPU en el tipo de entorno y volver a conectar.")
if not re.fullmatch(r"[A-Za-z0-9_-]+", EXPERIMENT_NAME):
    raise ValueError("EXPERIMENT_NAME: usar letras, números, guiones o guion bajo.")
if not MODELS or len(MODELS) != len(set(MODELS)) or not set(MODELS) <= {"yolo11n.pt", "yolo11s.pt", "yolo11m.pt"}:
    raise ValueError("Elegir modelos únicos de YOLO11n/s/m.")
if BATCH < 1 or EPOCHS < 1 or not 0 < DETECTION_CONF < 1:
    raise ValueError("Batch, épocas o confianza inválidos.")
gpu = torch.cuda.get_device_properties(0)
ENVIRONMENT = {"python": sys.version, "ultralytics": ultralytics.__version__, "torch": torch.__version__,
               "cuda": torch.version.cuda, "gpu": gpu.name, "vram_gib": round(gpu.total_memory / 1024**3, 2)}
print(json.dumps(ENVIRONMENT, indent=2))
EXPERIMENT_DIR = Path("/content/drive/MyDrive/GreenLight/experiments") / EXPERIMENT_NAME
EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)


## 4. Funciones de preparación y metadatos

Este bloque está incluido dentro del notebook: no hace falta clonar el repositorio.
Valida clase única, cajas, imágenes y correspondencia con los metadatos.


In [ ]:
"""Preparación de un export CVAT de una clase y particiones agrupadas para Colab.

Este módulo se incluye dentro del notebook para que no dependa del checkout.
"""

from collections import Counter
import csv
import hashlib
import io
import json
import math
from pathlib import Path, PurePosixPath
import random
import re
import zipfile

from PIL import Image
import yaml


IMAGE_SUFFIXES = {".png", ".jpg", ".jpeg", ".bmp", ".webp"}


def archive_members(archive):
    members = {}
    for item in archive.infolist():
        if item.is_dir():
            continue
        path = PurePosixPath(item.filename.replace("\\", "/"))
        if path.is_absolute() or ".." in path.parts or ":" in path.parts[0]:
            raise ValueError(f"Ruta inválida en ZIP: {item.filename}")
        if path.as_posix() in members:
            raise ValueError(f"Ruta repetida en ZIP: {item.filename}")
        members[path.as_posix()] = item
    return members


def indexed_members(members, suffixes):
    result = {}
    for name, item in members.items():
        path = PurePosixPath(name)
        if path.suffix.lower() not in suffixes:
            continue
        if path.stem in result:
            raise ValueError(f"Nombre base repetido: {path.stem}")
        result[path.stem] = item
    return result


def validate_vehicle_label(text, name):
    lines = []
    for number, line in enumerate(text.splitlines(), start=1):
        if not line.strip():
            continue
        fields = line.split()
        if len(fields) != 5 or fields[0] != "0":
            raise ValueError(f"{name}:{number}: se esperan cinco columnas y clase 0.")
        cx, cy, width, height = map(float, fields[1:])
        if not all(math.isfinite(value) and 0 <= value <= 1 for value in (cx, cy, width, height)):
            raise ValueError(f"{name}:{number}: coordenadas inválidas.")
        if width <= 0 or height <= 0 or cx - width / 2 < -1e-6 or cy - height / 2 < -1e-6 or cx + width / 2 > 1 + 1e-6 or cy + height / 2 > 1 + 1e-6:
            raise ValueError(f"{name}:{number}: caja fuera de imagen o sin área.")
        lines.append("0 " + " ".join(f"{value:.8f}" for value in (cx, cy, width, height)))
    return "".join(line + "\n" for line in lines), len(lines)


def metadata_from_csv(path):
    with Path(path).open(encoding="utf-8-sig", newline="") as source:
        return list(csv.DictReader(source))


def metadata_index(rows):
    result = {}
    for row in rows:
        filename = row.get("output_image") or row.get("filename") or row.get("image")
        if not filename:
            raise ValueError("El CSV necesita output_image, filename o image.")
        key = Path(filename).stem
        if key in result:
            raise ValueError(f"Metadatos repetidos: {filename}")
        match = re.search(r"(sesion_.+?)_(cam_[12])_frame_", key)
        session = row.get("session") or (match.group(1) if match else None)
        camera = row.get("camera") or (match.group(2) if match else None)
        if not session or not camera:
            raise ValueError(f"Faltan sesión/cámara para {filename}.")
        timestamp = float(row["simulation_timestamp"])
        if not math.isfinite(timestamp):
            raise ValueError(f"Timestamp inválido: {filename}")
        result[key] = {
            "session": session, "camera": camera, "simulation_timestamp": timestamp,
            "temporal_bin": int(row["temporal_bin"]) if row.get("temporal_bin") else None,
        }
    return result


def prepare_cvat_dataset(corrected_zip, destination, metadata_rows, images_zip=None):
    """Las etiquetas SIEMPRE salen de corrected_zip; el ZIP auxiliar solo aporta imágenes."""
    destination = Path(destination)
    if destination.exists():
        raise ValueError("El destino ya existe; usar una carpeta nueva.")
    metadata = metadata_index(metadata_rows)
    with zipfile.ZipFile(corrected_zip) as corrected:
        members = archive_members(corrected)
        configs = [item for name, item in members.items() if PurePosixPath(name).name in {"data.yaml", "data.yml"}]
        if len(configs) != 1:
            raise ValueError("Exportar CVAT como Ultralytics YOLO Detection: se necesita un data.yaml.")
        config = yaml.safe_load(corrected.read(configs[0]).decode("utf-8-sig"))
        names = config.get("names") if isinstance(config, dict) else None
        if names not in (["vehicle"], {0: "vehicle"}, {"0": "vehicle"}):
            raise ValueError("El export debe tener exactamente 0: vehicle.")
        corrected_images = indexed_members(members, IMAGE_SUFFIXES)
        label_members = {
            name: item for name, item in members.items()
            if PurePosixPath(name).suffix == ".txt" and "labels" in PurePosixPath(name).parts
        }
        label_index = indexed_members(label_members, {".txt"})
        listed_images = set()
        for name, item in members.items():
            if PurePosixPath(name).name in {"train.txt", "val.txt", "valid.txt", "test.txt"}:
                for line in corrected.read(item).decode("utf-8-sig").splitlines():
                    path = PurePosixPath(line.strip().replace("\\", "/"))
                    if path.suffix.lower() in IMAGE_SUFFIXES:
                        listed_images.add(path.stem)
        if not corrected_images and not images_zip:
            raise ValueError("El export no contiene imágenes. Indicar el ZIP original de imágenes en IMAGES_ZIP.")
        with zipfile.ZipFile(images_zip if not corrected_images else corrected_zip) as image_archive:
            image_index = indexed_members(archive_members(image_archive), IMAGE_SUFFIXES)
            expected = listed_images or set(image_index)
            if not expected <= image_index.keys():
                raise ValueError("Faltan imágenes nombradas en el export de CVAT.")
            if not label_index.keys() <= expected:
                raise ValueError("Hay etiquetas sin imagen correspondiente.")
            if not expected <= metadata.keys():
                missing = sorted(expected - metadata.keys())[:3]
                raise ValueError(f"Faltan metadatos: {missing}. Subir selection.csv de esta selección.")
            if not expected:
                raise ValueError("No se encontraron imágenes.")
            destination.mkdir(parents=True)
            (destination / "images/all").mkdir(parents=True)
            (destination / "labels/all").mkdir(parents=True)
            records = []
            for key in sorted(expected):
                image_bytes = image_archive.read(image_index[key])
                with Image.open(io.BytesIO(image_bytes)) as image:
                    width, height = image.size
                    image.verify()
                filename = PurePosixPath(image_index[key].filename.replace("\\", "/")).name
                text = corrected.read(label_index[key]).decode("utf-8-sig") if key in label_index else ""
                label, boxes = validate_vehicle_label(text, filename)
                (destination / "images/all" / filename).write_bytes(image_bytes)
                (destination / "labels/all" / f"{key}.txt").write_text(label, encoding="utf-8")
                records.append({
                    "filename": filename, **metadata[key], "width": width, "height": height,
                    "boxes": boxes, "image_sha256": hashlib.sha256(image_bytes).hexdigest(),
                    "label_sha256": hashlib.sha256(label.encode()).hexdigest(),
                    "missing_label_in_export": key not in label_index,
                })
    return records


def make_grouped_splits(records, seed=12345, mode="auto", gap_seconds=10.0, explicit=None):
    """Agrupa sesiones o tramos; excluye el margen entre tramos de splits diferentes."""
    if not math.isfinite(gap_seconds) or gap_seconds < 0:
        raise ValueError("El margen temporal debe ser finito y no negativo.")
    sessions = {row["session"] for row in records}
    if mode == "auto":
        mode = "session" if len(sessions) >= 3 else "temporal"
    if mode not in {"session", "temporal"}:
        raise ValueError("SPLIT_MODE debe ser auto, session o temporal.")
    result = [dict(row) for row in records]
    for row in result:
        if mode == "temporal" and row["temporal_bin"] is None:
            raise ValueError("La separación temporal necesita temporal_bin en los metadatos.")
        row["group"] = row["session"] if mode == "session" else f"{row['session']}:t{row['temporal_bin']:02d}"
    groups = sorted({row["group"] for row in result})
    if len(groups) < 3:
        raise ValueError("Se necesitan al menos tres grupos para train/val/test.")
    if explicit:
        assignment = dict(explicit)
        if set(assignment) != set(groups) or any(value not in {"train", "val", "test"} for value in assignment.values()):
            raise ValueError("GROUP_SPLITS debe asignar todos los grupos a train, val o test.")
    else:
        random.Random(seed).shuffle(groups)
        test_count = max(1, round(len(groups) * 0.15))
        val_count = max(1, round(len(groups) * 0.15))
        assignment = {group: "test" if index < test_count else "val" if index < test_count + val_count else "train"
                      for index, group in enumerate(groups)}
    boundaries = []
    if mode == "temporal" and gap_seconds:
        for session in sessions:
            spans = []
            for group in {row["group"] for row in result if row["session"] == session}:
                times = [row["simulation_timestamp"] for row in result if row["group"] == group]
                spans.append((min(times), max(times), group))
            spans.sort()
            for left, right in zip(spans, spans[1:]):
                if assignment[left[2]] != assignment[right[2]]:
                    boundaries.append((session, (left[1] + right[0]) / 2))
    for row in result:
        row["split"] = assignment[row["group"]]
        if any(row["session"] == session and abs(row["simulation_timestamp"] - boundary) <= gap_seconds for session, boundary in boundaries):
            row["split"] = "excluded"
    split_names = {row["split"] for row in result}
    if not {"train", "val", "test"} <= split_names:
        raise ValueError("Un conjunto quedó vacío. Cambiar grupos o capturar más sesiones.")
    # Evitar que dos copias idénticas terminen en conjuntos distintos.
    digest_splits = {}
    for row in result:
        if row["split"] == "excluded":
            continue
        old_split = digest_splits.setdefault(row["image_sha256"], row["split"])
        if old_split != row["split"]:
            raise ValueError("Una imagen idéntica aparece en conjuntos distintos. Revisar duplicados/grupos.")
    return result, assignment, mode


def write_dataset_split(destination, records, assignment, mode, seed, gap_seconds):
    destination = Path(destination)
    for split in ("train", "val", "test"):
        rows = [row for row in records if row["split"] == split]
        (destination / f"{split}.txt").write_text("".join(f"./images/all/{row['filename']}\n" for row in rows), encoding="utf-8")
    config = {"path": str(destination.resolve()), "train": "train.txt", "val": "val.txt", "test": "test.txt", "names": {0: "vehicle"}}
    (destination / "data.yaml").write_text(yaml.safe_dump(config, sort_keys=False), encoding="utf-8")
    with (destination / "split_manifest.csv").open("w", newline="", encoding="utf-8") as target:
        writer = csv.DictWriter(target, fieldnames=list(records[0]))
        writer.writeheader()
        writer.writerows(records)
    canonical = {
        "seed": seed, "mode": mode, "gap_seconds": gap_seconds,
        "images": sorted((row["filename"], row["image_sha256"], row["label_sha256"], row["split"]) for row in records),
    }
    fingerprint = hashlib.sha256(json.dumps(canonical, sort_keys=True).encode()).hexdigest()
    summary = {
        "dataset_fingerprint": fingerprint, "seed": seed, "mode": mode,
        "gap_seconds": gap_seconds, "groups": assignment,
        "images_by_split": dict(Counter(row["split"] for row in records)),
        "empty_images_by_split": dict(Counter(row["split"] for row in records if row["boxes"] == 0)),
        "missing_labels_in_export": sum(row["missing_label_in_export"] for row in records),
        "scope": "Independent capture sessions" if mode == "session" else "Internal temporal holdout; nearby scenes can remain correlated",
    }
    (destination / "split_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
    return destination / "data.yaml", summary


In [ ]:
import base64
import zlib
DEFAULT_METADATA = json.loads(zlib.decompress(base64.b64decode('eJy1fduuZTeu3a80/HwgiDdRyq8EgeEEPgcNtM9p9OUpyL+HXF0G4gmJa3Jm0q56sGsX1+aWRA7eBv/7//7pv/75j7/+8x8///m3X/7j15/+259+6r3Dz3//9e9//q///Bk7DugAP4NQF/lZUP6d4Of/9ctvP+PP//63X3779ef++WcMXO2v//kfP/3bn36yv+t/2WWFYvxrTdKvf/vFv/Qj81//629/+eVfwv3/u+SP1D//9s+//PIPF/ePP//269//8ctvf/U/pz5bn6OrsPCSTuRf/Y9ff/vrf/3tl7/8/D///PlG+Kf/829/2uqKN3SF/1dXQFr9/19X2Oj6ER0pO2g1GTAmIuOy72Wnqx51pbSugjqLdHXRka6TeiMyRZeJXcSy03UddeWsrn2qUI2uLjlSlXm1wYwEdriiQGOnqxx1lfR71akv3OHde3XJ4Xtd1OZksjssosTMOV1HVlcAnFyj60d0pKxMap0ABXFBF9re4fN71bxtsltcZZtMdGibkBuwjr46qyzQnK4zfYfXfMM27c7VJYfHCtAQRrfjpbVw/1zHUdWVvsKs9ok1V9hFR7qqOTu7uZMnzEnac2YYelpXM/mjSFcXHV7hOdriodrt+tp9xp2u86hrGjaB2CUu0tVFh+4V1Oww+u0F8zy0fa7nc83DJu5cBZtcdHiHGZuiydU1Qeb+Dp/P9Q5s+qNpglV2ris+Vns7dssRDDXR6AckAUdV76CmC/JXKFLVJYdIAriZqspTzLuOraZH3A95zMTERUbYJUeagkiDOVQGwlLgOXe64lHXNGbqZuylRleXHGJhO9XBsgaZv8HRt6oe4SHkIRNjfyF03dslEx3aJeG2Vl9sgELGlKRvvQOZLvDQoukqeGiiQ99qwZXhqmEWmAWh5+AhpDGTgVAoSkm45PC5zt7WmGTWF8lf7DYlcXyueAcz/TF0HUteCHF25+qSQyNsUfrkRTosxrEfes4K4x3I9EdV2X7XqOqSw2MlaRbejGUB3QLBvtX1fKx3INPFt643INPWt64YMREuO1aiqe5ZJ8PWNJ3PNZ1p6oZEX/Cuu3N1yaEVNshkPhXMAM8Ba59oOkImzCeaBmBRUs0lh+hQsS3lMQYoi/mDnap0VPUOZvqjqmrfUY2qLjm8wX6s2udaxO5g9w7nfIPvYKbrDZai3ItLDm/wdFUVDS9Z3GqwKXeD72CmPxomjy9qDJNLjv1NYwTDSyBsaHhus4dnVfNZJpFRlD10yeFj/fgbsTfLusTCj61hOr/WO5Dpj/CQDJwVwUMXHV5hmk0GTTOOU2ns/c0xRKcHkElGEex3yeEV7oaEufPqqtBnzrNSvjbHIlVxq0kOEZP9iMUMkx2sEJOuFGCiB4AJuSoZYZLDxzo9JTzFrfCac+TeKuUrc6xV9/cjOk4Jc7MYR5eKmSbpW3dzDFspn2N6x7Nu3U2pZ6UHSSakqseKFD9W9kK6+RnzAwr27/YKn19rvjBnf6cq0++iI2XFfStbRKcwUBZvs0zHCg7lEROXNUjwlwYJQ/ptGogQe3lDGLaI6XysecREcxYZYZccqmo/42GvbpmWuJjG9rmedc0X5uSVosahgBOH6BPIrvBCAUS7kbJNMh2tMKcLc31aDFWVE+6hFea+GjMPBVAcPMc2ljsmhTmPmYTlhbh1i/pNcggk0ICwBUFz4Zj2ZLeqHoEEP6jL4arLf68YSAxoIKtPnQKD98Wq8xXO1+UstKgqayjFZQ3zrmaahtjdWj9g8/0LnM8xkWpRRtglx0bYQnQic8GL7FfP+RtOQyZ7K1V1uY/o0Ab30WSuAWadFPtI2uB8Xc6i6CIY4ZJDxNSngX60e4wW9k1euaYXThfmumHuqn48HHE/nv2QlQm7GIowOJx7rOmynKcCihLCLjl8rMvzaYNMW7sAP+pa9x9rOsXUx+CqStXguFJlIALsBXkvk7cd5ipVksdLQ6syhy45VBXUwtYl9kiX9mzxRtJFOcPdXFZXXhzXlYdhQ0CY6neYci1bkk4xvdRhufU2tR2WkoZLht/eyDHtPSt+yTHJat2gviEmnCCYyzFJHjAZ4C6ywS459KyADdGiVpgs5l5zjlXSRTmAVdbx4qLDKwzcptpdHHN4F/pW1/MVzgMm5VEEDl1yaITHaB3J9ERxILxPCZ+tcDrHBNRHVdehiw7PVQ0IfxLCg7y7Z5tPO5vhfJLJcGiRrsxfkkxDGo8h7mDx96TqbcgkecikZhyKaugmObzCvD5FOR4s5l73kP94g0e6KAfQsQgdfkSHVthc3fAGNfXiv279zdEwjTxksidVpuqPoPs8lcLNbCnqpGmeNalqHjJZoPvCBd7n+TmO5IR7m2iAaXrT3r6h/+haRz7BhKOq1uqSQwvc2xJlgSVdZY79Yz2WqsaDqpz58SJdf4yZHHW1+8vDzhW9a3btG/HOquZTTPZWqlJMLjpMMS3PBxOy4Q01K7zvTutHZfOQacw3shHbuHXG2QjC3ngCWHiL/rT3UPjscPKQScre60d0eLBi+FDRY3S2gF22kCk42DxmUnxjwHWLhTEecCX7KXtjj5dv7P1N3SaFzwebL8yRB5BFWNhEh6ErepOlxaw+GmgocavrEQtrvpUJy/pJ8Us/KWEjr7NyF2+Inphrx9M8bOpSBYY/omMsQW0uNpezdPQffYq3wYTmK3PEZbq66PAO994IxJ4s6oJ95+H5Cudx0ygLXceXyBXtg6eSwUPT17ATbH3OseCq6UwTAL6Bm/bIH2PgJDrbgum0CosG7jHiEfprvp3JYGgVL8j60mVJYK9VDPM70UTfZySOHkcfZJoMyRV5V5Mcqqqe7jfDpMxCQ3OoSfPNTHNW9eO55EhVFmm6CEFYdG+WjoU5zRfmkHvVUAr3EB3y+Ewsq08vTDva3FCKPuhk4jcaBvZImOOOgWmOVWE6yQsZeNpX5s5IeOZLcxZKFbX38I/U0bkbT9vCwT7GO9jca649YuY5Bt5JNO3pTyoTTTMPmICr6ugf0bGqqxnaH8RjGrbag8Ozrvl5OcVVNG3kkuNgjpua27AAZwit5GDVTCeagOiNkc99LEfxzOdYaEG6lz/AYPDc9wkfgfB8UJt7hazoAA4ryYpmvv/b7k9V8sVFh+eqvTEPO1Ji1r7vxzufa76ZieWNzsNti4/EnYcwfLLKpHaL1Pu+aeDsb/IcA2YEq26wi45f62g+yTuRpmm7n+Q9n2q+NDdfucHbFrUvF9jL2lN1gOlssvfd30covPK0TKxvUAwcKKhijgEDwQ0QYS2GZRHC1rcee0FWnmNgzlV1rCY5jm+0TZxduMsyI7afDjyfax4zDX6Dbmv7XF10pKx3Pc6pE1eXzrg3TWfcvx6QDPQ3aG228VyPaW2czNIeqhLJnHQYhTzGcytfneNZNkg2vwySzdm8DWTYXf/wZOSmcFaeZWAY8i5qKkUK86T2VBuI4UOyKB1xjw+PycOVJxmQWUV/4pJDVQWat36Tio8UU1LVPGSioUURjksObzDMRk5+S8Bg8eu+hHO+wQ86wAcX3WCXHCckxJleLGwlcz5ZfLjyiabxCj7c19K/4MPVLXLVoTwNcVgkukX9R4cDPY+aUKmqDOmiQzAs08I5U7krO/NhqgwJtzjAL6ipl6Gm/gU1GZCgCT4dM5Z51xSlMOQpwLtF0lUV1xnzDCiaETaPs6YQ64/ekbs4AvIM4N3gWVX34fzSfbi6T+GYrfZZWJmpmTnIE4ADYRWL2kd0+Fjnat1T4MtZA2R/g8+PNd/RZF6tKAP+ER3pOu2nbEoaqLUwfYLmYD88YAD3RHuVy+F4ZmMBtc5LPKBTC1+3cXqg7JMu8KoZ14/oOKvGzVmNvGzl8ynJW/ygo6lsGEe/zOIQd6dkZdU5lnIyCw55DvBuuKWqe2tobIgtyOGB0icJ035K+6jpLQbwq3eFIpfjkkPv6oeqc0kXBOKVitIBHmCmV2hBtpjpCy0IMzZWbxCVMbq30qbca54A3CsKRZkmlxyGrubrfJpXlz1X7oeZ9FPsCnkCcCCtGk/5iI4RojR/RAYTuzOAp0pWcIsB/HKuOKpSEiY5PFcYzZ7pnDxkYe/b53o+1geoqYxf7CM6RE04G3p0MO1MgWeKagBuUYBfV4mUFdOxx8X0gb2ZQTLP6gxzc58EP1Ui4RYH+IVWAaqYqH4npj+aYXPqyIsWiNcj+77l/WyGn6Sa3uh92aeaKntf4BYH+DVbWkUh4ZLjbKk4Uak5JiZ09ssUjw/c4gC/9Pm8Mju37/MpnJ2DBxzg70DhbU6iEgrfYgC/TKV7gaFmKt0kx4bJObdoGG5immuffTl1u0OeAPylBsStv6ltQIQ8Bbj546qRjY/o0Ln23gQ/XANqIQulOtXgFgf4JQduz6UqB26iQ119QAVXp4li/mPPfXjWNV+eG1y1Q9Alhw5nSbO7i3a0ar/3TD5n05TfmzKgbDWkiw7f67JjJd+qaMGHSc+NzsEDFvBl/r4IIMqMAaKdq11gmBaveKNPih0E8iTgL9F3bsuupfSdkCcBB6e4qZomg5i3yOI9M8NM4IsElzn61LneogG/4H5+Y4Z5i/s5HmGGwc1rVjS6nevEfXXjfK55xoEhb+w52if8Jd5ztMDi9EWiFqTzxP064rNtesAELnNW0fm46NAQWyg5hM2xGka0eC43nA63qMAvykJV9PoRHWdgqHVD/iizz9NWp2MG5hYX+EXXpWUHu/TLwXprnmH/YUa5a7ZT4hYZ+CXd9EqNbp9u+lKjc0pCi/sG2plO75hIocRbZOAX6DSoSlcXHZonu8SeUvODtUhnb4qDg833g9vXVY27uujQx1I38+TfgkFFWPsBpLOPzWMn7lVDOS459LFOdGOIGAnZ7CpsH+zRxT5gAx/wRsfadrIX4o411M8mQWdWUFQ721w1J88GbgjkjS1Wh5XE8RqrMQ3+s0tWWLRWaqQBbtGBXzkfy7al85cBJB29gflX8IB9rP1QztHB5unAwbe9VcFEWHHKaVEzZ2OB9+LBfb+99myHHzCCT36DJntbauaYJduCnNa9k2t6VGeXIFdqzlOC259V1elccmiciJuCF6TH9CXMuZLkLUbwS4mD32DO2JY4+AtzhoXrwj4tKoPYnmzSDuc7wkWgqoNAIO4gIAt0sM/uZBKMfX+Fz7qmM04WCJbZJhcd+pzlAxyG+8FusaGmbenq7HPyLeG+kLLK54y4/qpztLnQQeI0Tfdc70efc4sV/JL516r3+hEdnmvXNhbZJe7D8NNe12Oc84QW/BXavH1M94U2b5D5V1+pbYE3ZDXNL56TuYqaLl1yaJlEG+tUNFVNeoq9CPKc4L6fsYqpCeL0MKq27t1bhvs925RaXgu3KMEvFfVX6CT2FfVKOgmQByN0sypydcmRqoTD91iBCbZgiCVFkw0PKMEHVfGWuuTwBlswR8MQP/kClX2J7nyB863gAG+Mu+4vMMTzruK9Pn0o+GIRzlaZ84Tg4OWiKgxBccpfhx1rN+dLIor79pczhHhACO4D8TXtL7q+tL9Mj27ET3/y2Bc3joFcnhDc3ipVrYv5MSZ2fqvqa6zWZ9TKC3S5Bq4HjOAoZZP4LjrOgLsNHkwWs8KCHFMTPKAEF/ugqkbaCXHmZdjB2vWF6bYpmwC/xQn+xzsM841l6dtznfGydG+PXhbC2Zfpmn0eipFHXfOsA2twkcdxyWGSyYLWJbP7pmlnzEuxs0KeE7zPV7aobEkQvyxRYZltsi41NXUeNoifrXA+x6TMRZZJf+zhOi9R4WZ316s37MWNZMflA0Zw5rK8i4sOcYSPz83OC5Gc7CbFcwO3CMGv9JarqLvUJYfwEJzGU2ksRgA8bGM7W6Y0aAKCKuKMj+jQu+psg83bmHUiJshN5NwiBL9sZmAs6rh0yaFp0uaJJXFrrcg5f/OADlxWmW+VFTvXqcPv8DIjgzwO7JZnIHGLD/zicbpWba/tGjvX7gzDAuiLxBdmXc4DQvB3WOS2S8S/kIS4nZhT/APEu332zNHnS/xgV6/ZwKKOS5Mc60ptraUqFtOhLNlf4qOu+WZw4rKUhIsO3Wv3NsTVyY508SHIOZvhB7BpVXW+uOT4vfYGZFCYnKBA5vYKn59rvqOJhKs6mlx0fKyzmVi0e7ycVCLXIZEnBQcnSyrS1UXHmSb2aUEwNGyQmHJbMeEBKzgNKptjHhTPMQM3Zy5C0ikkySrkA1Lw0d/oytvvoIib8nCM5tkIe6zmdBT2SYljrilPCu79NUXn6pJDl4P2XMVjnMFmin8449udardYwS8R3SxzOTxjlzO7kyxMO3+QPtZ+eOP4Xh+wggtTVYcE0xeOBU+rfbZ3qdgl2Oeaznf4Qa4J3xje2OaaMJ7d4M/CDdTPbKQ93q2qx1zTLVbw6+auWaSqSw4TMEJtzDm7gHeEjNwWW7jFCn7pF9Yy9+qiw6yESDMj3D+Q6Xejff9c87wDQlV7KFxy/Fyp/Ys5gybRwhSbJ9yiBb+oanFjkaq4YlV9Ise5ovocunDfGn1W9UGq6RXQtAfDpaDpFi34ZZRBqGoex0VHui4Ydq6AOoD60kOy6ZiAecILPlYZW9P6wtY0mhM++tDrUDPFuQTMLV7wP77Xd6pWW9RUW7V6QAvuNL9FCRiIW6OngeHlExu+Y7sfyhtnVfPzcwOqmEs/okNdV2+4wDMvHufsGVGC95pvajI0WrV0wySHd3hqs9jGvGxHBei5cZw8L/hnSq+qeeAHueGxvDEMDZOZJQPDOkaScOwWMfgFDcssorbXHyyz50177MTgY67pvPZre6xHMPyAF9yTWkX8Lyv2OALqGwt8MSbPTltNj1A4Twvel92domiONGZYmNwGgP9yIi7ez3+edMU8K3gf9qsq09TjKH0YPhwG/FGRaO5HrE5YGPOk4NDrkqX9S7JUPqM4s6PwMBSxp949H+uDpqZVxmzvokPfqr43Bu3RMiEemNWOvhVv8YJfMoj4hm/dZhAx9q3LOQdUvQ98gq79EtsTZsIHvOD8yhLbfcL/yxZbg79mm5yfUM33jC1kOuUP8RYt+JVL4o2+vAOXRGFfHt5iBb/oSlXcah/Roa6kzWkzuhpCVN6Tv5zPNT85905tY9tHW1rbwDwpuIUXZWMbiF+IS8lCnDlpLpPu1OA7XU99tJgnBYcxpIzoZkicfnECI3YeAHO0MPb7Rc4P9hYv+KUY6Zi0qBgp8KXIPBor4SKcSzjXO4CQpxzodSOR/ctIpAg056TyeB6H7kc3jrjpFjH4BUr43sYaKDFWqCrAaoTD2xCJZUmONgPzxOAGRrEINrnkGA6bHbb4ZnZ2FysHyoGjrvm+JvNvVcErfOmQdiL/D7k9G/Yf+2bwU7YUbxGDX7HEG+tUDljiyzoVog9ucibdYceU6hrGPDG4LzEsOlb5stIKWRv4ro3lqWFnCkxhiVvE4BfoD2WlHIYvpRxebfmmPXH0P/dkiOdzfUAMrlJFEKgStyH65unhdIwWw3rROEX8iA+IwQnecDn7jH/scha0QWyAyfnmFux5lY9W+AEvuOd6ytpfMEZNazYycMgKKA4mUlf4ATH40Kp2cJccZoZhNAJ7rmywyXwOp/YBYZ4aHGCWtVy66DDMAWg6xvIdT6AHNHwMc25xg18z/lUJU5ccnutYzRCz/Uhw+O6C5LE+AE34xnT6FjRhPJ1uVtrAsGfBDSGqaIppGPPE4E4GW2SZXHLocCxqNj09zUTD14ZudT06nDwxOHTpRTPbH9Eh8GdsU5zJf6kpzbnyxi1m8MucCr7BW7qdU8GYttTnnry8QYBLVHJdEviAF5xeGcXfhq5fRvEBuTmumkPILLDkdhRjnhjcvP0bLIh7LDxjFkQ1Q4GmqK7h1Cn7brUjkLhFDH6dAu11U6D9CwuigSbx69vlkwvP6fpgm8qqyoO75PC5dmhAgmv1wbjnQzm+1lu04Ffy6DK6d/lC9z7NMvl6YvKOtXlIqp1PNd/TNBSKIleXHENhbgZolaedLOxd6xEx3aIEv3aWvrEuct9ZGq+L9A/2H7QTKhti2nvWYzriFiP4JXsIb1C9b7OHEDO9k5i3GTLUDZN0TG3GxAd84H28kSjdA6bxJVNqQaTvvRZCi3IsoEsBplt84JfwZlZtiXfJ4bEuH8QfXTxidVb7HGK6RQd+0VXfWBuz1VXjrTE0tQ1fAeoZYfOt+2mcs655xDS0bFmBiw7TTIR2sAYMPxMqtO9UO1fnbhGCX9KHUpY+lC/pQ4vRRbvT0Ipv8NrapiMSvsUHfskemkMuyh7+WBhyPla7wp5Oc/4XmPv+/qMZfkAHbrisqg7pokNdu++gMDu8LMI1eJjrF8Y8H3ifIlWBq0gcuJJnShc4brPfa8+HcrRND+jALYoqQv0uOXyu5MQZwg7hndp+T7Bwfq8PEk2vrD7aJppKVx/hAz7wSW/sZdjOqFC8l4GR7b36q/a9R7z2Y5HnO5zvappSFc+55Pi9aiOUNTxPSr/vl7yva76ryUBiVRnHRYd4WLwVfHgCZjlTbK6R9hYf+GWAbr7B5bkfoJtfuDypN++2hU8IMinFJoG3+MAvmSZ5gyRkn2mSmCVEdTX1WRzPgh9M0zElkacD7yRvDOJvYZPEc/hggTosXELKZK5vD/2PZvgRIfgbm6f3hOCVm6fxFiX4JTFsl6cqMWyiwys8vatJ0EywTN+4l7vD+WSTAe+i8NUlh6piQ4cSa/og6A/6+9vQXx4sUdE3Rsq2V1jjiTLDeK3zmgC4DMfnqHfxASW4QNWKYpccqmpBjvOBr+WlOV9BndM1D5qI31hVsLXCHK8qANPVfvu6IDtc0X2F+WyF81RN9sOvmtxw0SFoWtwsoDPfyjo9Q7zT9dgj8YAVHH0fTw2ZJ2KcRNTGy+cEwZDion0d52iZ8qzg9jFvVOf2dRyJq3PTTJNXIc1PMi/CZF7tAS04eTNrERg20aHL6dS4G2oyE0x6GMQ/guFbtOCXpMTiqqSESQ5tkxlFtsPFqetDi5JLStyiBb9MC75SjNxPC5YWI2/Rgl98zqg6V5ccI//RcKjzvki3O4zbO3w+1zxssmCqKInokmP/Kr6g2FlazYzRgQrxbIjTuAkGvJEIPyxQjDPhvvTaglAAA08WxO5bQgJDnK/SEVRVrlxynAlvFgfJWBbS65p723Q+1ydkTaMKD5vkEA87+8u/iKlkYe+5Dv9bvODXTp+6/At/y7+g72Iekzt33S8+OlrhJ7Tg9AZC3MauFCNE83QyDQd72zuute+iPV7gW7zgF8PUy7bYuujYMNkNBlGQSWLR694IHw3TLV7wCxweVaSPH9GhsvJxOT7J7Gz++zscKPuAd0Cqun1ccuxeVwM0/94tYHcW8Vyd+RYx+HVLTlW7z0d0eLDqLAtsHod9194BSpwPNj9C566tKosocduwzt4Up4zpbJ5zH8CeLfEDZnCpYnx3yXGs3hsYWiJfy+Ys6NuM6bE97xY1+GUJnbzBnrFfQicxe4YsbSzTUIQFACz7+PWYg8lTg/c535hT2RYkZzynYpFr0z7HEDPJiDPXS3uLGfxao9MyxncTHT5XgoawnC0dyB5TLi1xixn88ly7FOnqksPnStPgsE8f6VA+tEocX+sDYvB3VN2259Wqmi/RvcOttmUcK+RWw1us4JfegVdmyva9A1+GygS1maZOBmDude7D9LOueczUX+lr2uv6pbHJYrlm/tUch9c3mLZjkWdd85DJu7iqeqR7nFcTZ85Y7HsxeSzeY+Gzrg/6mlbVxPZc8cQ2D/JVtl2nsOKBR+7oWm+xgl/cDVXR0brk0Aab8dfpRJ72Us1KbW/w2QbnE03Yq9okXHKYaFo+ZTV8w+vwue1De+lR1wcMl1zF1eSSwxvsa3vt/k6TjIL7BunjY33ACo69bDYd+5fZdMTWfS0bkXiHRw7z3yIFvzicWcb66KJDIzygscVkFt4uIu7bIP18rg820HEVtZpLjjOIs5mddkIq/X23wW0b/IAS3PlRi6rpGrc0DWndbO8A8X/mvqXpaJduUYJf+PLGGx3De768EbcMO3vdIrGbO6U7P2DusT4ozr3C9L6v4RQyveMtRvBL3uUVfpB93qWUH+QWI/gFCMMb6f49EIY43e9L20UdAndxXvAcu8ItSvALjJhvLLLdwogZ77GVrs2rGmaTeHTZV3HOquZrc++Q2u8tUyWpPd3iBL/MCnJVjukjOtJ1QTffaq7DA7q19rMMx1w/5VnB7c+qKAdccgj7ffjok3ZRu8jYYetyTgVmusUKfsHCWhW4fkTHlxiafRGYeaIPyULuEuc7wUcvo8tz0bF/pebd65OG/SuSvcT5VnAt4xnWLzTDxAYm2JdBKRuo0FSqlG7Rgl98Tl9FsMklx8mX1QAsvOnKnVlym6fpFi34ZTnDmEXH6pLDYwVsoKbqVDth4lT2hW6xgl/qy1LWviXypX3LwOnsw7eFykTc5w+D55ofoONexTL8ER0p6xsNzddYiOOlSNzvFTzb4Xyq6R2K1v3oRiVFK+VZwe0OVbXlueQ4hdibb4zx0QQcul+eflY1D5ssVC6CEi45ziBSE8P85OMjA3DfNnDC/pQnBQesm4vEL3ORw9zrJCf/9c6RfaR+il4pzwlufrxsQbGLDi0TuBkGX8c8px52sh0t0y1O8IvLUSzbkfNj7c15wQg0HN6rb66Voef2RdIDVvDhs5hFcBgl3jy9PuuYmTsNi1/3/SCBsg820L0Ch/dxTuXaDXpACy5alm9y0fEtluYwYuAy/9TTtzgPnMjLDFVrN0aI/rU7s496Mf2zKz5nnfIJJ1pVmwU/okNVLXIWw8N9gMH/sc9LHHW9xQt+mbV6hX13O2tVyr5LD3jBGbTovbrkUFcfZXZ6tWna/h7s3lc1j5vMCFaNlbno+Apjw7U+NCE4eG19zvkK53vBLb4oAhMuOU5LSFuDDUcsBgvrtxmYY1biFi34ddHTLJu0+sHocl70NJpB/wWMHz7EXCs43SIGvxJKvGGGD4QSX8zw1GYh3TJ77VlTTQ3l0C1i8Ev4OqtI0F1ymG6yc1U2TS2sQ7NTW1WP6aY8L7j9SN9gCDx415gi0Dxq40E4cU071P0+5rNperBMZVIVG+KkUFWY08Ic8mIkGB62j8q5nDxq8mbBqkpzj8fxTcs2DCwRT18Wv2+lPSL/W8TglyhHypC/i47TEqtNC+gtqPeZ13359azrg84mLmsMcdHhudLyRkTndRoKsI/ojummW9TgF4rA9QZV65YicMVMrWD2X8YQMMBkaCKJEG8xg1+Sw6NsuHd8Ge5laEvmsBer4gW9FLEP3eIGv2RgxhsdBPsMzIg7CBZCY6cttaiO0XxsDjbdYge/9KwRFFXVXXIIh8HC9GlfNWSqb6RIMQ/QA3pw8wxVVXUXHdompTaRvZvWznhxitmHHtCDm5+qIh6gOLFmpkmV1CscznF/2PF6VDUPm8zaF2VMXXJsm9AiujnHcsbhyftlZWfblCdssgCjCCK65PAGd2l2y3XIYjFDtn+uR/f6gBycZtViNpcculf0VdvdcMRgH+HYj28c/esDdnCkWcRe6pJDMyxtLBj2L6u5HUnxhNAtdvDLc11vsINvn+uKycFRVhNvkO/SB5/GN47P9RY5+EVXeCPZtNUV4mQTGoaZa3ULcmX1sedVPqua72yyWKqqzX/FJXUe2AxB0AAftvJdganE2gNu8CFSFLyOH01ox/xLN10noskfgmPfJX3Mv9yiBr80rL0yQrdtWPs2Qmeqwlxj2atGmvtGxPOp5lNNZhPKJl5/ZEDPbNm9gYWgbHfYp8tynRJPmMGxrFPCRcdhem9D/FtQIVj7K3zWNY+ZZi9jt+9f2O0NM6F+VsUrT5z7DpjjHX7ADI5aNsnsosNUk6g307on4GH/kRrgoFvU4Nf0S1VbiEuO0y++R5GGc2ECmjVOET/SLWrwS2qYZhFocsmhrjIb4bJg3q6w2ufksPAtavBLyr+OIJC+EAQOJ/Xxrae47LVub/Ax4X+LGPxK6VO1/ki/rD8iZwYX7/lUHcOJAlNAQh6soeMqzOSSQyu8mrKK+hZQQdX9htdjiHOLGvxyrkBF0ZxLDs/VXJ051WXfwpzOrZw71nxXE/WqlZEf0eFjVWpOW0Q6eNBhP8P5ueZBE8Ab6wX3NSv4sl5QuRmAYE8BcR97Cumzc31Qn1MsmwRVjMfoDDUZXpZunsmg0z6cO+qapwbvk98AEluAyDGQYMPC9oYQndJzwEiNqNMtYvBLlE5l28roy7YyeztzwXBW8LHGyM3k5HnBYVDZDJ2LDlPgfbQ5xCC/7z3t2bmyJ8TgUsWJ8hEdKetZWgOG4qu+7HN6VtknBbpRNjE4RlzcsFssvqnMoqG5dB/QBco+6GuabzCO7fuaZkw5Nrs52O77j5yVGPbI/9jXdIsa/EpKW7Vm5CM61NXsIiyDTKqC68dq+cTBPugHxze2Au2VxXgtkCGJhr47aHq0xfvlbOeDfcANPt8AxPvNVjEgRjFALIPZou7uu15zOf88ObihCS0yxS45RhP0mQV1Qk+e42Cdjj42zw4OBr6rurhcdIgSx2c9m5rC+Nkjk0L/t+jBL3whs44vZH7hCxnY2DtL4UNnlEqY5rnBu31KlWUyyeFrHY08n8ZLDMhDjvCGbjGDX5c9vUE0vG8JqSQaplvM4BckAVWtPh/Rka7qk9sGmqmrv9k9wdr5seZR0zvjkfu0ROl45ANi8On7L2rCV5McOhz7YAtbyVuaPBOeIpCmW7zgl/aXKUWWySWHr9XeDkKXD4+Rerk5Z4UfYCb7qqr9gj1MDjvz7zTxFsupDEi2SdziBb/QLMCoSja56PC1CngL11CGzr+3Z96GwnlecOC6+V7+Mt+r9nj6AGcdc0e8p0U565pvBn+H0nMPDyspPekWM/gFSbyy+XSPJL6sPrV36ruevMd/sLc25XTNd4OrIdGiphDSuG5lYbp3+UwwEGGOZz/KfCxwPOEGX2WDKi46hv3U2BC/xXLm5X8YstsuJ88N3hdwFVeISQ6RhNcjSXj0xdPrV7kerjw3uFn7sn2gLjo8V9KmdqQ6ZTDOfdb/fK4PhujK0i8f0aHPQfc5BhDNV3qlI4f8b7GDX9JqvSx4ddFhWs2R/8SFZE4WaZ9qOvvXfGeTvDJEt20v/TJDZ0/UXI5bR5giOvaMw0eMeIse/NJzyW/QBG57LjlmCcRl7tUZQoYun0HbUw8cfc4tevBr/8Aq6lhzyaF/NcBmCMIeNRtmEpLtHT7rmuceEMKylbaE8UpbMv9qcZ8BVRiedEq91wcE4b6/vChSl3j9ke+XNWtqAc4aNHA/B3p0r3mCcOdPr9obuWI2OfsRkwF+WKbscNOUyiE+4Qfvb8ypbEetejyn4gRr5lQViX3zxr4N8axqnh+cX1kec+BDjJfHKGMzRbsvswV7szmumyf84K90wOwB4pcOGB8vQN/Za0e6JuzJlY8A8RY/+HWs7A0jvB8ri22wdGrOJeGedXXfIJPTNb+Eji1ErkrAmOjYNEGzEN3tsOhcOYDIeYLw7oXPqtFIjuePzFAsRDtPUl269ku8TgCRb/GDXwHiG9s39gAx3r5hHqkNcpIQWPalvG+UOIEmfsAPLnVkN/KF7MYz8DDVLLAfbd+P5RwbJThPEO6j3kWX2CWHUU7vFtCJf4CIGaf9fsGTg+UH/OBTqzJrLjlEiOJjOcPsdTfkv/YdpieEyHl+cPDJiarEGnxxOuiJNfet01e05Yhp+RY/OFxsU1UvokuObdNsOgCWc/pjkrCJb/GDX9yrQhkPl4kO3evqjiUmT5hmjPcQ8exeHzSEjyqShY/o8Aqv1ZxVeZh7NeG5nD/n6cEBuG4lM3/rlMBGBuEXT7W4bu9ej7rm6cEBaZVN0ZnoSFfDEU3sWJ3f3kDxnnTsrGseNhkAL6rRueTQNHVw6oHFPqLpTaY5iJjnB39pLmdbt6qcy+EH/OBmBquusIuOsxLOnrE8ykU71rUtbxzN8C1+8Eu6lKt481xyeKy+5lXNdahFdf0Q0J2P9cEU3St0Tdu0WildE98iB7+OINUto9Mv3QN9GBh2loU+fOFrascgPyAH9y6bqpGGHw085yVIsyH4fit/q7qvbpwDugfk4PZ3qjg9XXQIEX15pE4L1MfytaCpznfOs4P7+GjRJXbJYfDqm4GmGPR3MlJZqRl1vkUOfk3AVFGF1AY5D7jBBd8gzN7m1TDmy0aw12ouzzDTUIvSt/jwiJlucYNfa8xvzG3va8zx3LbJbkp9DelqT1b3k2VnXfPd4L4araprWGMGGEXvQzRNcdoR477ueipv8C1y8AtomlqUBnfJIWha1AwEk8U4Ikq87x04pppucYNfmEJeWTG4ZQr5smEQvNVnicfoRDT3HdJnI5xva+K6tib+0takYzYSU9Yetrnhfcvl0bc+4AY3/12VQXTRka7LAkldZk87E1rMvjXDZ9CUJwcHkjKESBIjRDU0zOokMDiXHFYzn21THjS9w3azfa+lbDd8ixz8utyqihflIzpE/ha9kn42WwkzUW6YmW+xg18my+asusRzxskmaJ+ed7NiMBfAntLzGL4+YAd/ZyxnW8qpHMvhW+zg19HIqjLzR3SYGx69sQ9TW1Q3l27t8LGSkycH70uqukJccniqFnLMPvpai9X7m1KdEnyLG/zK/VhFxOWSY4joXcOsKMuz4CvVrsZ5avC++I2M//ZYOU74+wfrGIi4hHTtCznnU80zg4v9fIoS/kxxmI6f/YITnGTByX1SpLR8ixr8suwJq9bQueTwWHU2g3mEBGyYeE+rfD7WB5ipV+0E4v5lNRt7l4QKEjIQ71MSR8h0ixj8Apleyb7sIdOX9Itv6CFhJ0tAp8HLlZhvEYNfbLC+Afu3Nlhj1E/eXWomGJ1NpI+5T0kcjfAtZvBLR8gre3L2HSFf9uSYos3ZzCcNtUswczjiATP4KBtTGV+mVFC9M090EjuzMmtqaQHfoga/ONdRtXfDJcfOdbTl5Ho6xcuoqaFtvsUMfm30qVpo65LjHDg2E+7rbNfCvr/Bxxz4LWbwSw+ivEH4vu9BlJjx3Q6zLepoH4Lq+zdygesDbvDR35ja3qea+petzGYUu/rmGGcdo30dJ1D2QappvYEQ96mmFUNEJTHYZMGK1zfkB0P87VTTLXLwSy1ylbU1uejwYM1UDJ+gG0rK++neI5TIc4PbDxSruB8HxtyPjI29V3j4ti/7pBT5Iz/hBkeoWs2GEK9mcyTB5lgXEntBPVd3fcANPlYVMZVLDiEiYfOOd6cvWgq0N8RHp5PnBvehpaKUhEsOz5WoKXo+zbvG9TDJfD7XBzSXWJYsddGhaVqfLsRJuuT3FZP3XY48YB6wi1+Vf5F4/9GcTf/lX00s6jZZeoSID8jBxyyDiDOGiIS9fbIvw4xY57FfV3Z+rnnYxKOqufQjOkQS0xmz7QKjuR1G2eLhs3vNdzX5sruqrATML3yeBhEN+Xuuf/mi15yuT2boVl2VecVVZuUm05d/oDfU5oa2Oc8ODki9iiLQRUe6DkfDvrXAKS5/r/rcbkN8QA9unq2ovdQlh+4VRvOGvIXgfeMz6V7z/OAvZda2DTC1mbVb9OAXcqoyopDabducZwf3Mb2iUo5LDqEESBti2F/7miq8r+WczzVPc+n5gCrbpDHNwnAyOVo+I8Y+lL/V9RipPyAH9ybWqqwExpSeY0HrwzukddKiPZY465rnuZSlVU2XS+Omy4HN94oLA5CFdrmZnFvU4Jdjret8py+d70OXQX+a3StXZplSlBKcpwaHdyaZ9zxcX0aZxxoNfDDSVyku5pyuD6jBYb0Bm/bTG+sLbAKD/t5xOeaHZS0Hm/LU4J7uKTJN/GWZIvjiSMMRc/0LNI1ctukBOTiOqt1WLjlEEqstR7W+TLOj7JtCjh1ct8jBLxHdqiL2+YgOIzodTZ2+1Fn8vTkkl4G5xQ5+aZWQqllmlxzCJvOuusS7/OWzoi3XApNnB/dmqLJtivKlv9TCnO5EgkJoyu4z4ef3ml/gu8Ysypi65PC9Gg5f03nk7FyXGalc9TVPD24O/40emAOWiJtgxuS2iOa0sK4v2DeEn/1rnrJJXhmi20LELzN0CHauPrehfXbt+6zEESI+oAc3t1Zlh110nIGZTVDtu3bix7U/1iOUuEUP/sfnCkuqFnmZ5PBYuRn4hw+HkYV0eyRxdK8P2MGHvamqmrqJjnRdTj3ghXezTgYS9+RUZ/d6ix78MritVcORLjlG/s0OFXQsAcOI+07E87nma3QOzYpUldgIi/PvChGw/4OYQ4h5cvCXmAe23XmlzAMPuMG9G6OobCXxGjrqo5m5MBPsxPO0X9FwVvXJ+t5etr63f6EJgYYWlTk/waCh+x7/s675TBOOqgVILjmO0g0d+piuim9Q5z3z4/m15lNNFmgU0Sp/RIfxXGfDEc4y7ORqh21lx6rVLWrwSwOMvvFctw0wGj9XmMMJAocCosiek/YY4NwiBr8Svr9Rs9r3IX7hpJ12n5w0Q0xVX1OVm3jNE4Ob/35jt+BhQD1eLqjoDocUP/20wrkh0FvE4BeH8wo83DqcL+gQ7QYbUhp2tIaZhCFHPPCAGNxcXFFPCPV4oa2FrUJ2j9XtEu+3Mh9t8C1e8Iu/kTd2qG/9jXxZoU5twiL3OT6dLgeim6OuDyATVL1WlxzeYDXIBAum743Ruad7P1/gPFcT8hu9avsQneNeNac4HkoDzTpNWnuWkGOIfosY/FrZmHWVjfkFM63mczL+UqfIfj/mWdd8lskn/osME8aqWrhs59mn7+xdUzn5WvNZpvEKwcI+HfGFYGGBNGAnMBo+U5bM9sstYvBLL7hgUSVy/sjfnzdRjIYW3U4i32d8JH/5H/8XBk67iQ==')).decode())
print(f"Metadatos incluidos: {len(DEFAULT_METADATA)} imágenes")


## 5. Cargar el export corregido y separar datos

La separación es reproducible y se guarda con hashes de imágenes y etiquetas.
Para comparar 500 contra 1.200 imágenes de entrenamiento, conservar el mismo
conjunto independiente de validación/prueba mediante grupos explícitos.

Un TXT ausente se interpreta como imagen negativa, según el formato YOLO.
Revisá el conteo de etiquetas ausentes para detectar exportaciones incompletas.


In [ ]:
LOCAL_ROOT = Path(tempfile.mkdtemp(prefix="greenlight_", dir="/content"))
def local_copy(path, filename):
    source = Path(path)
    if not source.is_file():
        raise FileNotFoundError(f"No existe {source}. Subir el archivo a Drive y corregir su ruta en el paso 2.")
    target = LOCAL_ROOT / filename
    shutil.copyfile(source, target)
    return target

corrected_zip = local_copy(CVAT_ZIP, "cvat_corregido.zip")
images_zip = local_copy(IMAGES_ZIP, "imagenes.zip") if IMAGES_ZIP else None
metadata_rows = metadata_from_csv(local_copy(METADATA_CSV, "selection.csv")) if METADATA_CSV else DEFAULT_METADATA
DATASET_DIR = LOCAL_ROOT / "dataset"
records = prepare_cvat_dataset(corrected_zip, DATASET_DIR, metadata_rows, images_zip)
records, group_assignment, used_split_mode = make_grouped_splits(records, SEED, SPLIT_MODE, GAP_SECONDS, GROUP_SPLITS)
DATA_YAML, split_summary = write_dataset_split(DATASET_DIR, records, group_assignment, used_split_mode, SEED, GAP_SECONDS)

record_df = pd.DataFrame(records)
display(record_df.groupby(["split", "camera"]).agg(images=("filename", "count"), vehicles=("boxes", "sum")))
display(record_df.groupby(["group", "split"]).size().rename("images").reset_index())
print(json.dumps(split_summary, indent=2))
if used_split_mode == "temporal":
    print("Evaluación interna por tramos: no demuestra generalización a sesiones nuevas ni equilibrio por clima.")

TRAIN_CONFIG = {"models": MODELS, "imgsz": IMGSZ, "batch": BATCH, "epochs": EPOCHS,
                "patience": PATIENCE, "seed": SEED, "workers": WORKERS, "optimizer": OPTIMIZER,
                "lr0": LR0, "ultralytics": ultralytics.__version__}
experiment_meta = {"dataset_fingerprint": split_summary["dataset_fingerprint"], "training": TRAIN_CONFIG}
metadata_path = EXPERIMENT_DIR / "experiment.json"
if metadata_path.exists() and json.loads(metadata_path.read_text()) != experiment_meta:
    raise ValueError("Este experimento tiene otros datos o parámetros. Usar otro EXPERIMENT_NAME.")
metadata_path.write_text(json.dumps(experiment_meta, indent=2))
stamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_%f")
(EXPERIMENT_DIR / f"environment_{stamp}.json").write_text(json.dumps(ENVIRONMENT, indent=2))
for filename in ("split_manifest.csv", "split_summary.json", "data.yaml", "train.txt", "val.txt", "test.txt"):
    shutil.copyfile(DATASET_DIR / filename, EXPERIMENT_DIR / filename)
# Copia compacta de TODAS las etiquetas corregidas; imágenes conservadas en tu ZIP de Drive.
with zipfile.ZipFile(EXPERIMENT_DIR / "corrected_labels.zip", "w", zipfile.ZIP_DEFLATED) as archive:
    for label in sorted((DATASET_DIR / "labels/all").glob("*.txt")):
        archive.write(label, label.relative_to(DATASET_DIR).as_posix())
print("Dataset local:", DATA_YAML)
print("Resultados/checkpoints:", EXPERIMENT_DIR)


## 6. Inspección visual de las etiquetas corregidas

Las cajas mostradas aquí vienen del export CVAT. Esta celda no ejecuta YOLO.


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
sample = record_df[record_df.split != "excluded"].sample(n=min(6, len(record_df[record_df.split != "excluded"])), random_state=SEED)
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
for ax, (_, row) in zip(axes.flat, sample.iterrows()):
    image = Image.open(DATASET_DIR / "images/all" / row.filename)
    ax.imshow(image)
    for line in (DATASET_DIR / "labels/all" / (Path(row.filename).stem + ".txt")).read_text().splitlines():
        _, cx, cy, width, height = map(float, line.split())
        ax.add_patch(patches.Rectangle(((cx-width/2)*image.width, (cy-height/2)*image.height), width*image.width, height*image.height, fill=False, edgecolor="lime", linewidth=1))
    ax.set_title(f"{row.camera} · {row.split} · {row.boxes} vehículos")
    ax.axis("off")
for ax in axes.flat[len(sample):]:
    ax.axis("off")
plt.tight_layout()
plt.show()


## 7. Piloto: medir costo antes del entrenamiento completo

Usa cinco épocas del primer modelo, el mismo train/val y ningún dato de test.
La proyección usa el promedio de las épocas posteriores al arranque; no estima
el costo de modelos de distinto tamaño ni garantiza el tiempo del entrenamiento.


In [ ]:
COMMON_TRAIN = dict(data=str(DATA_YAML), imgsz=IMGSZ, batch=BATCH, device=0, workers=WORKERS,
                    seed=SEED, optimizer=OPTIMIZER, lr0=LR0, amp=True, plots=True)
pilot_summary_path = EXPERIMENT_DIR / "pilot_summary.json"
if RUN_PILOT and not pilot_summary_path.exists():
    pilot = YOLO(MODELS[0])
    torch.cuda.reset_peak_memory_stats()
    started = time.perf_counter()
    pilot.train(**COMMON_TRAIN, epochs=5, patience=5, close_mosaic=0,
                project=str(EXPERIMENT_DIR / "pilot"), name=stamp)
    elapsed = time.perf_counter() - started
    epoch_log = pd.read_csv(Path(pilot.trainer.save_dir) / "results.csv")
    epoch_log.columns = epoch_log.columns.str.strip()
    per_epoch = float(epoch_log["time"].diff().dropna().mean())
    pilot_summary = {"model": MODELS[0], "epochs": len(epoch_log), "wall_seconds": elapsed,
                     "mean_epoch_seconds_after_first": per_epoch,
                     "estimated_full_minutes_same_model": per_epoch * EPOCHS / 60,
                     "peak_allocated_gpu_gib": torch.cuda.max_memory_allocated() / 1024**3}
    pilot_summary_path.write_text(json.dumps(pilot_summary, indent=2))
    del pilot
    gc.collect()
    torch.cuda.empty_cache()
if pilot_summary_path.exists():
    print(pilot_summary_path.read_text())
else:
    print("Piloto desactivado.")


## 8. Entrenar o continuar YOLO11n/s/m

Los modelos se entrenan uno por vez, con los mismos datos y parámetros. `best.pt`
se selecciona usando validación. Al cortar Colab se conserva lo escrito hasta la
última época guardada en Drive; lo que estaba calculándose puede perderse.
Una GPU distinta al reconectar puede cambiar tiempos y algunos resultados.


In [ ]:
if RUN_TRAINING:
    for weights in MODELS:
        model_name = Path(weights).stem
        run_dir = EXPERIMENT_DIR / "training" / model_name
        complete = run_dir / "completed.json"
        checkpoint = run_dir / "weights/last.pt"
        if complete.exists():
            print(model_name, "ya completado; se conserva el resultado.")
            continue
        started = time.perf_counter()
        if checkpoint.exists() and RESUME_INTERRUPTED:
            # Ultralytics finaliza/limpia last.pt al completar un entrenamiento.
            # Recuperar el marcador si Colab se cortó entre esa limpieza y nuestra escritura.
            checkpoint_info = torch.load(checkpoint, map_location="cpu", weights_only=False)
            finalized = checkpoint_info.get("epoch", 0) == -1
            del checkpoint_info
            if finalized and (run_dir / "weights/best.pt").is_file():
                complete.write_text(json.dumps({"model": weights, "dataset_fingerprint": split_summary["dataset_fingerprint"],
                                                "recovered_finalized_checkpoint": True}, indent=2))
                print(model_name, "ya finalizado; se recuperó su marcador.")
                continue
            model = YOLO(str(checkpoint))
            print("Continuando:", checkpoint)
            model.train(resume=True, data=str(DATA_YAML), device=0, workers=WORKERS)
        elif run_dir.exists():
            raise ValueError(f"{run_dir} está incompleto. Activar RESUME_INTERRUPTED o usar otro experimento.")
        else:
            model = YOLO(weights)
            model.train(**COMMON_TRAIN, epochs=EPOCHS, patience=PATIENCE,
                        project=str(EXPERIMENT_DIR / "training"), name=model_name)
        actual_dir = Path(model.trainer.save_dir)
        best_checkpoint = actual_dir / "weights" / "best.pt"
        if (actual_dir.name != model_name or actual_dir.parent.name != "training"
                or not best_checkpoint.is_file()):
            raise RuntimeError(f"Revisar el destino del entrenamiento y best.pt: {best_checkpoint}")
        run_dir = actual_dir
        complete = run_dir / "completed.json"
        complete.write_text(json.dumps({"model": weights, "dataset_fingerprint": split_summary["dataset_fingerprint"],
                                        "wall_seconds_this_connection": time.perf_counter() - started,
                                        "finished_utc": datetime.now(timezone.utc).isoformat()}, indent=2))
        del model
        gc.collect()
        torch.cuda.empty_cache()
else:
    print("Entrenamiento desactivado. Activar RUN_TRAINING para generar best.pt.")


## 9. Comparar en validación

mAP usa `conf=0.001` para construir las curvas de precisión/recall. La precisión
y recall resumidos por Ultralytics corresponden a su punto de máximo F1.
El MAE de conteo usa, por separado, `DETECTION_CONF` fijo y cuenta toda la imagen.

La latencia mide el detector con batch 1, calentamiento y sincronización CUDA.
Incluye preprocesamiento/inferencia/postprocesamiento; excluye lectura de PNG,
CARLA, ZMQ y DeepSORT. No representa los FPS del pipeline completo.


In [ ]:
def count_and_benchmark(model, split):
    subset = [row for row in records if row["split"] == split]
    output = []
    first = np.asarray(Image.open(DATASET_DIR / "images/all" / subset[0]["filename"]).convert("RGB"))[:, :, ::-1].copy()
    predict_options = dict(imgsz=IMGSZ, conf=DETECTION_CONF, iou=0.7, classes=[0],
                           agnostic_nms=True, rect=False, device=0, verbose=False)
    for _ in range(3):
        model.predict(first, **predict_options)
    for row in subset:
        frame = np.asarray(Image.open(DATASET_DIR / "images/all" / row["filename"]).convert("RGB"))[:, :, ::-1].copy()
        torch.cuda.synchronize()
        started = time.perf_counter()
        prediction = model.predict(frame, **predict_options)[0]
        predicted_count = len(prediction.boxes)
        torch.cuda.synchronize()
        latency_ms = (time.perf_counter() - started) * 1000
        output.append({"filename": row["filename"], "camera": row["camera"], "group": row["group"],
                       "true_count": row["boxes"], "predicted_count": predicted_count,
                       "absolute_count_error": abs(row["boxes"] - predicted_count), "detector_ms": latency_ms})
    return pd.DataFrame(output)

def evaluate_models(split):
    comparison = []
    for weights in MODELS:
        name = Path(weights).stem
        run_dir = EXPERIMENT_DIR / "training" / name
        if not (run_dir / "completed.json").is_file():
            raise FileNotFoundError(f"Completar el entrenamiento de {name} antes de evaluar.")
        model = YOLO(str(run_dir / "weights/best.pt"))
        if model.names != {0: "vehicle"}:
            raise ValueError("Esta evaluación necesita el modelo entrenado de una clase vehicle.")
        metrics = model.val(data=str(DATA_YAML), split=split, imgsz=IMGSZ, batch=BATCH,
                            device=0, workers=WORKERS, conf=0.001, iou=0.7, rect=False,
                            plots=True, project=str(EXPERIMENT_DIR / split), name=name, exist_ok=True)
        per_image = count_and_benchmark(model, split)
        per_image.to_csv(EXPERIMENT_DIR / f"{split}_{name}_per_image.csv", index=False)
        camera_report = per_image.groupby("camera").agg(images=("filename", "count"), count_MAE=("absolute_count_error", "mean"))
        camera_report.to_csv(EXPERIMENT_DIR / f"{split}_{name}_by_camera.csv")
        comparison.append({"model": name, "split": split, "images": len(per_image),
                           "precision_max_F1": float(metrics.box.mp), "recall_max_F1": float(metrics.box.mr),
                           "mAP50": float(metrics.box.map50), "mAP50_95": float(metrics.box.map),
                           "count_MAE": float(per_image.absolute_count_error.mean()),
                           "detector_ms_median": float(per_image.detector_ms.median()),
                           "detector_ms_p95": float(per_image.detector_ms.quantile(0.95)),
                           "detector_FPS_mean": 1000 / float(per_image.detector_ms.mean()),
                           "count_conf": DETECTION_CONF, "dataset_fingerprint": split_summary["dataset_fingerprint"]})
        del model
        gc.collect()
        torch.cuda.empty_cache()
    table = pd.DataFrame(comparison)
    table.to_csv(EXPERIMENT_DIR / f"{split}_comparison.csv", index=False)
    return table

if all((EXPERIMENT_DIR / "training" / Path(weights).stem / "completed.json").exists() for weights in MODELS):
    validation_comparison = evaluate_models("val")
    display(validation_comparison)
else:
    print("Todavía faltan entrenamientos completos; la evaluación no se ejecutó.")


## 10. Prueba final reservada

Fijá resolución, modelos y umbral usando validación antes de activar
`RUN_FINAL_TEST=True`. Evitá ajustar parámetros mirando el resultado de test.
Se comparan los modelos entrenados sobre exactamente el mismo conjunto reservado.

Para comparar también pesos COCO originales hay que implementar un evaluador
que filtre car/motorcycle/bus/truck y remapee predicciones a vehicle. No se incluyen
en esta tabla porque su numeración de clases difiere de la del dataset.


In [ ]:
if RUN_FINAL_TEST:
    test_comparison = evaluate_models("test")
    display(test_comparison)
else:
    print("Test reservado. Activar RUN_FINAL_TEST cuando hayas fijado la configuración y ejecutar esta celda.")


## 11. Descargar modelos y reportes

Cada modelo está en `training/<modelo>/weights/best.pt`. Los archivos también
quedan en Drive. El ZIP de entrega incluye mejores pesos, tablas, particiones y
parámetros; los checkpoints para continuar entrenamiento quedan en Drive.

Al integrar `best.pt` en GreenLight, adaptar el filtro de clases a `0: vehicle`.
La evaluación de seguimiento y conteo por ROI en CARLA es un paso posterior.


In [ ]:
delivery_zip = EXPERIMENT_DIR / "greenlight_models_and_reports.zip"
with zipfile.ZipFile(delivery_zip, "w", zipfile.ZIP_DEFLATED) as archive:
    for weights in MODELS:
        best = EXPERIMENT_DIR / "training" / Path(weights).stem / "weights/best.pt"
        if best.is_file():
            archive.write(best, best.relative_to(EXPERIMENT_DIR).as_posix())
    reports = list(EXPERIMENT_DIR.glob("*.csv")) + list(EXPERIMENT_DIR.glob("*.json")) + list(EXPERIMENT_DIR.glob("*.yaml")) + list(EXPERIMENT_DIR.glob("*.txt"))
    for report in sorted(reports):
        archive.write(report, report.name)
print("Entrega:", delivery_zip)
from google.colab import files
files.download(str(delivery_zip))


Documentación: [Colab](https://research.google.com/colaboratory/faq.html),
[Entrenamiento](https://docs.ultralytics.com/modes/train/),
[Evaluación](https://docs.ultralytics.com/modes/val/),
[Exportación CVAT](https://docs.cvat.ai/docs/dataset_management/export-datasets/).
